In [2]:
import json
import os
with open('/home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct-thought/test.4.8,11:10-10.json', 'r') as f:
    data = json.load(f)

# Create a directory for level-specific files if it doesn't exist
output_dir = '/home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct-thought/levels'
os.makedirs(output_dir, exist_ok=True)

# Filter data by levels and save separate files
for level in range(1, 6):
    level_data = [item for item in data if item.get('level') == level]
    
    # Save level-specific data
    output_file = os.path.join(output_dir, f'level{level}.json')
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(level_data, f, indent=4, ensure_ascii=False)
    
    print(f'Saved {len(level_data)} items to level{level}.json') 


Saved 11 items to level1.json
Saved 25 items to level2.json
Saved 19 items to level3.json
Saved 22 items to level4.json
Saved 23 items to level5.json


In [3]:
import json
import numpy as np

# Load data for each level
levels = {}
for level in range(1, 6):
    with open(f'/home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct-thought/levels/level{level}.json', 'r') as f:
        levels[level] = json.load(f)

# Calculate and print statistics for each level
print("Level | Mean Accuracy | Mean Validity | Count")
print("-" * 50)
for level in range(1, 6):
    data = levels[level]
    mean_accuracy = np.mean([item['per_question_mean_accuracy'] for item in data])
    mean_validity = np.mean([item['per_question_mean_validity'] for item in data])
    count = len(data)
    print(f"{level:5d} | {mean_accuracy:.4f}    | {mean_validity:.4f}    | {count:5d}")

Level | Mean Accuracy | Mean Validity | Count
--------------------------------------------------
    1 | 0.8182    | 0.9273    |    11
    2 | 0.9160    | 1.0000    |    25
    3 | 0.8842    | 0.9947    |    19
    4 | 0.9227    | 0.9727    |    22
    5 | 0.7087    | 0.8391    |    23


In [4]:
import json
from collections import Counter
import re

def is_valid_token(token):
    # Check if the token contains at least one alphabet character
    return bool(re.search('[a-zA-Z]', token))

function_words = {
    # Articles
    'a', 'an', 'the',
    # Pronouns
    'i', 'you', 'he', 'she', 'it', 'we', 'they', 'me', 'him', 'her', 'us', 'them',
    'my', 'your', 'his', 'her', 'its', 'our', 'their', 'mine', 'yours', 'hers', 'ours', 'theirs',
    'this', 'that', 'these', 'those', 'who', 'whom', 'whose', 'which', 'what',
    # Prepositions
    'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'from', 'about', 'between', 'among',
    'through', 'during', 'before', 'after', 'above', 'below', 'under', 'over', 'into', 'onto',
    # Auxiliary verbs
    'be', 'am', 'is', 'are', 'was', 'were', 'been', 'being',
    'have', 'has', 'had', 'having',
    'do', 'does', 'did', 'doing',
    'can', 'could', 'will', 'would', 'shall', 'should', 'may', 'might', 'must'
}

def is_content_word(token):
    # Check if token is valid and not a function word
    return is_valid_token(token) and token.strip().lower() not in function_words

# Initialize counters for the entire dataset
all_spike_and_fall_tokens = Counter()
all_fall_and_spike_tokens = Counter()

# Process each level
for level in range(1, 6):
    # Load data for this level
    with open(f'/home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct-thought/levels/level{level}.json', 'r') as f:
        data = json.load(f)
    
    # Process each question in the level
    for question in data:
        # Process each output (0-9) for each question
        for i in range(10):
            spike_and_fall_key = f'output_entropy_spike_and_fall_{i}'
            fall_and_spike_key = f'output_entropy_fall_and_spike_{i}'
            
            if spike_and_fall_key in question and question[spike_and_fall_key] is not None:
                spike_and_fall_tokens = [token.lower().strip() for token in question[spike_and_fall_key]['middle_tokens'] if is_valid_token(token)]
                all_spike_and_fall_tokens.update(spike_and_fall_tokens)
                
            if fall_and_spike_key in question and question[fall_and_spike_key] is not None:
                fall_and_spike_tokens = [token.lower().strip() for token in question[fall_and_spike_key]['middle_tokens'] if is_valid_token(token)]
                all_fall_and_spike_tokens.update(fall_and_spike_tokens)

# Get top 200 for each category
top_spike_and_fall = all_spike_and_fall_tokens.most_common(200)
top_fall_and_spike = all_fall_and_spike_tokens.most_common(200)

# Find tokens that appear in both
common_tokens = set(dict(top_spike_and_fall).keys()) & set(dict(top_fall_and_spike).keys())
both_counts = {token: (all_spike_and_fall_tokens[token], all_fall_and_spike_tokens[token]) 
              for token in common_tokens}
top_both = sorted(both_counts.items(), key=lambda x: max(x[1]), reverse=True)[:200]

# Convert data to JSON-friendly format with rankings
spike_and_fall_dict = {token: {"rank": idx+1, "count": count} for idx, (token, count) in enumerate(top_spike_and_fall)}
fall_and_spike_dict = {token: {"rank": idx+1, "count": count} for idx, (token, count) in enumerate(top_fall_and_spike)}
both_dict = {token: {
    "rank": idx+1, 
    "spike_and_fall_count": spike_and_fall_count, 
    "fall_and_spike_count": fall_and_spike_count,
    "max_count": max(spike_and_fall_count, fall_and_spike_count)
} for idx, (token, (spike_and_fall_count, fall_and_spike_count)) in enumerate(top_both)}

# Save to JSON files
output_dir = '/home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct-thought/'
with open(f'{output_dir}top_spike_and_fall_tokens.json', 'w') as f:
    json.dump(spike_and_fall_dict, f, indent=2)

with open(f'{output_dir}top_fall_and_spike_tokens.json', 'w') as f:
    json.dump(fall_and_spike_dict, f, indent=2)

with open(f'{output_dir}top_both_tokens.json', 'w') as f:
    json.dump(both_dict, f, indent=2)

# Print results with rankings
print("Top 200 spike and fall Tokens:")
for idx, (token, count) in enumerate(top_spike_and_fall, 1):
    print(f"#{idx}: '{token}': {count}")
    
print("\nTop 200 fall and spike Tokens:")
for idx, (token, count) in enumerate(top_fall_and_spike, 1):
    print(f"#{idx}: '{token}': {count}")
    
print("\nTop 200 Tokens appearing in both (token: spike_and_fall_count, fall_and_spike_count):")
for idx, (token, (spike_and_fall_count, fall_and_spike_count)) in enumerate(top_both, 1):
    print(f"#{idx}: '{token}': {spike_and_fall_count}, {fall_and_spike_count}")

print(f"\nTotal unique tokens in spike_and_fall: {len(all_spike_and_fall_tokens)}")
print(f"Total unique tokens in fall_and_spike: {len(all_fall_and_spike_tokens)}")

print(len(all_spike_and_fall_tokens))
print(len(all_fall_and_spike_tokens))

Top 200 spike and fall Tokens:
#1: 'the': 153
#2: 'is': 95
#3: 'x': 71
#4: 'i': 64
#5: 'so': 58
#6: 'a': 55
#7: 'to': 54
#8: 'that': 50
#9: 'and': 47
#10: 'of': 44
#11: 'let': 43
#12: 'me': 31
#13: ''s': 31
#14: 'but': 30
#15: 'it': 29
#16: 'wait': 28
#17: 'by': 27
#18: 'in': 26
#19: 'which': 25
#20: 'b': 24
#21: 'be': 22
#22: 'if': 19
#23: 'are': 19
#24: 'can': 18
#25: 'for': 18
#26: 'at': 17
#27: 'this': 15
#28: 'as': 14
#29: 'k': 14
#30: 'because': 14
#31: 'c': 14
#32: ''t': 13
#33: 'both': 13
#34: 'right': 13
#35: 'y': 13
#36: 'think': 12
#37: 'since': 12
#38: 'then': 12
#39: 'number': 11
#40: 'each': 11
#41: 'have': 11
#42: 'all': 11
#43: 'we': 11
#44: 'maybe': 10
#45: 'n': 10
#46: 'with': 10
#47: 'now': 10
#48: 'would': 10
#49: 'terms': 10
#50: 'should': 9
#51: 't': 9
#52: 'z': 9
#53: 'f': 9
#54: 'equal': 8
#55: 'equation': 8
#56: 'times': 8
#57: 'seems': 8
#58: 'original': 8
#59: 'just': 8
#60: 'first': 8
#61: 'point': 7
#62: 'angle': 7
#63: 'make': 7
#64: 'alternatively': 7
#65

#spike and fall

In [17]:
import json

target_tokens = ['that', 'so', 'let', 'me', 'by', 'and', 'maybe', 'wait', 'this'] #fall and spike thought transition
#target_tokens = ['that', 'so', 'let', 'me', 'by']

# Load the data
with open('/home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct-thought/top_fall_and_spike_tokens.json', 'r') as f:
    data = json.load(f)

# Print header
print(f"{'Token':<10} {'Rank':<6} {'Middle Count':<12}")
print("-" * 40)

# Extract and print info for each target token
for token in target_tokens:
    if token in data:
        info = data[token]
        print(f"{token:<10} {info['rank']:<6} {info['count']:<12}")
    else:
        print(f"{token:<10} {'N/A':<6} {'N/A':<12}")

Token      Rank   Middle Count
----------------------------------------
that       6      56          
so         5      63          
let        12     31          
me         17     25          
by         16     26          
and        10     39          
maybe      19     21          
wait       13     28          
this       58     8           


In [18]:
import pandas as pd
import json
import matplotlib.pyplot as plt

def analyze_row(row, level):
    trials = sum(1 for key in row.keys() if key.startswith('Output_'))
    question_id = row.get('id', 'unknown')
    
    token_counts = {
        'that': 0,
        'so': 0,
        'let': 0,
        'me': 0,
        'by': 0,
        'and': 0,
        'maybe': 0,
        'wait': 0,
        'this': 0,
        'step': 0,
        'steps': 0,
        'since': 0,
        'wait': 0,
    }
    
    # Lists to store correctness for aha and no-aha cases
    aha_correct = []
    no_aha_correct = []
    total_aha_trials = 0
    total_no_aha_trials = 0
    
    # Analyze each trial
    for i in range(trials):
        output = row.get(f'Output_{i}', '').lower()
        # Count individual tokens
        for token in token_counts.keys():
            token_counts[token] += output.count(token)
            
        aha_count = sum(output.count(word) for word in token_counts.keys())
        metrics = row.get(f'Metrics_{i}', {})
        is_correct = metrics.get('math_equal', False)
        
        if aha_count > 0:
            total_aha_trials += 1
            aha_correct.append(is_correct)
        else:
            total_no_aha_trials += 1
            no_aha_correct.append(is_correct)
    
    # Add token frequencies to question_stats
    question_stats = {
        'question_id': question_id,
        'level': level,
        'total_aha_trials': total_aha_trials,
        'total_no_aha_trials': total_no_aha_trials,
        'aha_correct_count': sum(aha_correct),
        'no_aha_correct_count': sum(no_aha_correct),
        'all_aha': (total_aha_trials == trials),
        'no_aha': (total_no_aha_trials == trials),
        'mixed_aha': (not (total_aha_trials == trials)) and (not (total_no_aha_trials == trials))
    }
    # Add token counts to stats
    question_stats.update(token_counts)
    
    return question_stats, token_counts

# Collect token frequencies across all levels
total_token_counts = {
        'that': 0,
        'so': 0,
        'let': 0,
        'me': 0,
        'by': 0,
        'and': 0,
        'maybe': 0,
        'wait': 0,
        'this': 0,
        'step': 0,
        'steps': 0,
        'since': 0,
        'wait': 0,
    }

# Process each level
for level in range(1, 6):
    level_token_counts = {
        'that': 0,
        'so': 0,
        'let': 0,
        'me': 0,
        'by': 0,
        'and': 0,
        'maybe': 0,
        'wait': 0,
        'this': 0,
        'step': 0,
        'steps': 0,
        'since': 0,
        'wait': 0,
    }
    # Load and process data
    with open(f'/home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct-thought/levels/level{level}.json', 'r') as f:
        data = json.load(f)
    
    level_data = []
    for row in data:
        stats, tokens = analyze_row(row, level)
        level_data.append(stats)
        # Update token counts
        for token, count in tokens.items():
            level_token_counts[token] += count
            total_token_counts[token] += count
    
    # Create bar plot for this level
    plt.figure(figsize=(10, 6))
    plt.bar(level_token_counts.keys(), level_token_counts.values())
    plt.title(f'Token Frequencies - Level {level}')
    plt.xlabel('Tokens')
    plt.ylabel('Frequency')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(f'/home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct-thought/level{level}_token_frequencies.png')
    plt.close()

# Create overall bar plot
plt.figure(figsize=(10, 6))
plt.bar(total_token_counts.keys(), total_token_counts.values())
plt.title('Overall Token Frequencies')
plt.xlabel('Tokens')
plt.ylabel('Frequency')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('/home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct-thought/total_token_frequencies.png')
plt.close()

# Print token frequencies
print("\nToken Frequencies by Level:")
for level in range(1, 6):
    print(f"\nLevel {level}:")
    for token, count in level_token_counts.items():
        print(f"{token}: {count}")

print("\nTotal Token Frequencies:")
for token, count in total_token_counts.items():
    print(f"{token}: {count}")


Token Frequencies by Level:

Level 1:
that: 5068
so: 11930
let: 3616
me: 7338
by: 1799
and: 7460
maybe: 1301
wait: 4166
this: 1800
step: 568
steps: 86
since: 1169

Level 2:
that: 5068
so: 11930
let: 3616
me: 7338
by: 1799
and: 7460
maybe: 1301
wait: 4166
this: 1800
step: 568
steps: 86
since: 1169

Level 3:
that: 5068
so: 11930
let: 3616
me: 7338
by: 1799
and: 7460
maybe: 1301
wait: 4166
this: 1800
step: 568
steps: 86
since: 1169

Level 4:
that: 5068
so: 11930
let: 3616
me: 7338
by: 1799
and: 7460
maybe: 1301
wait: 4166
this: 1800
step: 568
steps: 86
since: 1169

Level 5:
that: 5068
so: 11930
let: 3616
me: 7338
by: 1799
and: 7460
maybe: 1301
wait: 4166
this: 1800
step: 568
steps: 86
since: 1169

Total Token Frequencies:
that: 14876
so: 29677
let: 10086
me: 21985
by: 7127
and: 16801
maybe: 3301
wait: 8777
this: 4761
step: 2092
steps: 346
since: 3299


In [19]:
#aha and no-aha
import pandas as pd
import json

def analyze_row(row, level):
    trials = sum(1 for key in row.keys() if key.startswith('Output_'))
    question_id = row.get('id', 'unknown')
    
    aha_correct = []
    no_aha_correct = []
    total_aha_trials = 0
    total_no_aha_trials = 0
    
    for i in range(trials): 
        output = row.get(f'Output_{i}', '').lower()
        aha_count = sum(output.count(word) for word in target_tokens)
        metrics = row.get(f'Metrics_{i}', {})
        is_correct = metrics.get('math_equal', False)
        
        if aha_count > 0:
            total_aha_trials += 1
            aha_correct.append(is_correct)
        else:
            total_no_aha_trials += 1
            no_aha_correct.append(is_correct)
    
    all_aha = (total_aha_trials == trials)
    no_aha = (total_no_aha_trials == trials)
    mixed_aha = (not all_aha) and (not no_aha)

    question_stats = {
        'question_id': question_id,
        'level': level,
        'total_aha_trials': total_aha_trials,
        'total_no_aha_trials': total_no_aha_trials,
        'aha_correct_count': sum(aha_correct),
        'no_aha_correct_count': sum(no_aha_correct),
        'all_aha': all_aha,
        'no_aha': no_aha,
        'mixed_aha': mixed_aha,
        # Initialize metrics columns with None
        'aha_accuracy': None,
        'aha_precision': None,
        'aha_recall': None,
        'aha_f1': None,
        'no_aha_accuracy': None,
        'no_aha_precision': None,
        'no_aha_recall': None,
        'no_aha_f1': None,
        'aha_only_accuracy': None,
        'no_aha_only_accuracy': None
    }

    def safe_div(n, d): return n / d if d > 0 else None

    if mixed_aha:
        tp = sum(aha_correct)
        tn = sum(no_aha_correct)
        fp = total_aha_trials - tp
        fn = total_no_aha_trials - tn

        precision = safe_div(tp, tp + fp)
        recall = safe_div(tp, tp + fn)
        f1 = safe_div(2 * precision * recall, precision + recall) if precision and recall else None
        
        tp2 = sum(aha_correct)
        tn2 = sum(no_aha_correct)
        fp2 = total_aha_trials - tp2
        fn2 = total_no_aha_trials - tn2

        precision2 = safe_div(tp2, tp2 + fp2)
        recall2 = safe_div(tp2, tp2 + fn2)
        f1_2 = safe_div(2 * precision2 * recall2, precision2 + recall2) if precision2 and recall2 else None

        question_stats.update({
            'aha_accuracy': safe_div(tp, total_aha_trials),
            'aha_precision': precision,
            'aha_recall': recall,
            'aha_f1': f1,
            'no_aha_accuracy': safe_div(tn, total_no_aha_trials),
            'no_aha_precision': precision2,
            'no_aha_recall': recall2,
            'no_aha_f1': f1_2
        })
    elif all_aha:
        question_stats.update({
            'aha_only_accuracy': safe_div(sum(aha_correct), total_aha_trials)
        })
    elif no_aha:
        question_stats.update({
            'no_aha_only_accuracy': safe_div(sum(no_aha_correct), total_no_aha_trials)
        })
    return question_stats

all_trial_data = []
level_dfs = {}

for level in range(1, 6):
    with open(f'/home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct/levels/level{level}.json', 'r') as f:
        data = json.load(f)

    level_data = []
    for row in data:
        level_data.append(analyze_row(row, level))
    
    level_df = pd.DataFrame(level_data)
    level_dfs[level] = level_df

    level_mixed = level_df[level_df['mixed_aha']]
    level_all_aha = level_df[level_df['all_aha']]
    level_no_aha = level_df[level_df['no_aha']]

    print(f"Level {level}:")
    print("Mixed:")
    print("  Avg aha accuracy:", level_mixed['aha_accuracy'].mean())
    print("  Avg aha precision:", level_mixed['aha_precision'].mean())
    print("  Avg aha recall:", level_mixed['aha_recall'].mean())
    print("  Avg aha f1:", level_mixed['aha_f1'].mean())
    print("  Avg no-aha accuracy:", level_mixed['no_aha_accuracy'].mean())
    print("  Avg no-aha precision:", level_mixed['no_aha_precision'].mean())
    print("  Avg no-aha recall:", level_mixed['no_aha_recall'].mean())
    print("  Avg no-aha f1:", level_mixed['no_aha_f1'].mean())

    print("All 10 aha:")
    print("  Avg accuracy:", level_all_aha['aha_only_accuracy'].mean())

    print("All 10 no-aha:")
    print("  Avg accuracy:", level_no_aha['no_aha_only_accuracy'].mean())

    # Save level-specific CSV
    level_csv_path = f'/home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct/level{level}_analysis.csv'
    level_df.to_csv(level_csv_path, index=False)
    print(f"Level {level} analysis saved to {level_csv_path}")
    
    # Add to all trial data
    all_trial_data.extend(level_data)

# Calculate combined metrics
all_df = pd.DataFrame(all_trial_data)
all_mixed = all_df[all_df['mixed_aha']]
all_aha = all_df[all_df['all_aha']]
all_no_aha = all_df[all_df['no_aha']]

print("\nCombined analysis saved to /home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct/all_levels_analysis.csv")
print("Mixed:")
print("  Avg aha accuracy:", all_mixed['aha_accuracy'].mean())
print("  Avg aha precision:", all_mixed['aha_precision'].mean())
print("  Avg aha recall:", all_mixed['aha_recall'].mean())
print("  Avg aha f1:", all_mixed['aha_f1'].mean())
print("  Avg no-aha accuracy:", all_mixed['no_aha_accuracy'].mean())
print("  Avg no-aha precision:", all_mixed['no_aha_precision'].mean())
print("  Avg no-aha recall:", all_mixed['no_aha_recall'].mean())
print("  Avg no-aha f1:", all_mixed['no_aha_f1'].mean())
print("All 10 aha:")
print("  Avg accuracy:", all_aha['aha_only_accuracy'].mean())
print("All 10 no-aha:")
print("  Avg accuracy:", all_no_aha['no_aha_only_accuracy'].mean())

# Save combined analysis
all_df.to_csv('/home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct/all_levels_analysis.csv', index=False)

Level 1:
Mixed:
  Avg aha accuracy: nan
  Avg aha precision: nan
  Avg aha recall: nan
  Avg aha f1: nan
  Avg no-aha accuracy: nan
  Avg no-aha precision: nan
  Avg no-aha recall: nan
  Avg no-aha f1: nan
All 10 aha:
  Avg accuracy: 0.8181818181818182
All 10 no-aha:
  Avg accuracy: nan
Level 1 analysis saved to /home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct/level1_analysis.csv
Level 2:
Mixed:
  Avg aha accuracy: nan
  Avg aha precision: nan
  Avg aha recall: nan
  Avg aha f1: nan
  Avg no-aha accuracy: nan
  Avg no-aha precision: nan
  Avg no-aha recall: nan
  Avg no-aha f1: nan
All 10 aha:
  Avg accuracy: 0.9159999999999999
All 10 no-aha:
  Avg accuracy: nan
Level 2 analysis saved to /home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct/level2_analysis.csv
Level 3:
Mixed:
  Avg aha accuracy: nan
  Avg aha precision: nan
  Avg aha recall: nan
  Avg aha f1: nan
  Avg no-aha accuracy: nan
  Avg no-aha precision: nan
  Avg no-aha recall: nan
  Avg n

In [19]:
#many aha and little aha
import pandas as pd
import json

def analyze_row(row, level):
    trials = sum(1 for key in row.keys() if key.startswith('Output_'))
    question_id = row.get('id', 'unknown')
    
    many_aha_correct = []
    little_aha_correct = []
    total_many_aha_trials = 0
    total_little_aha_trials = 0
    
    for i in range(trials): 
        output = row.get(f'Output_{i}', '').lower()
        many_aha_count = sum(output.count(word) for word in target_tokens)
        little_aha_count = sum(output.count(word) for word in target_tokens)
        metrics = row.get(f'Metrics_{i}', {})
        is_correct = metrics.get('math_equal', False)
        
        if many_aha_count > 100: #a random threshold of 100
            total_many_aha_trials += 1
            many_aha_correct.append(is_correct)
        else:
            total_little_aha_trials += 1
            little_aha_correct.append(is_correct)
    
    all_many_aha = (total_many_aha_trials == trials)
    all_little_aha = (total_little_aha_trials == trials)
    mixed_aha = (not all_many_aha) and (not all_little_aha)

    question_stats = {
        'question_id': question_id,
        'level': level,
        'total_many_aha_trials': total_many_aha_trials,
        'total_little_aha_trials': total_little_aha_trials,
        'many_aha_correct_count': sum(many_aha_correct),
        'little_aha_correct_count': sum(little_aha_correct),
        'all_many_aha': all_many_aha,
        'all_little_aha': all_little_aha,
        'mixed_aha': mixed_aha,
        # Initialize metrics columns with None
        'many_aha_accuracy': None,
        'many_aha_precision': None,
        'many_aha_recall': None,
        'many_aha_f1': None,
        'little_aha_accuracy': None,
        'little_aha_precision': None,
        'little_aha_recall': None,
        'little_aha_f1': None,
        'many_aha_only_accuracy': None,
        'little_aha_only_accuracy': None
    }

    def safe_div(n, d): return n / d if d > 0 else None

    if mixed_aha:
        tp = sum(many_aha_correct)
        tn = sum(little_aha_correct)
        fp = total_many_aha_trials - tp
        fn = total_little_aha_trials - tn

        precision = safe_div(tp, tp + fp)
        recall = safe_div(tp, tp + fn)
        f1 = safe_div(2 * precision * recall, precision + recall) if precision and recall else None
        
        tp2 = sum(many_aha_correct)
        tn2 = sum(little_aha_correct)
        fp2 = total_many_aha_trials - tp2
        fn2 = total_little_aha_trials - tn2

        precision2 = safe_div(tp2, tp2 + fp2)
        recall2 = safe_div(tp2, tp2 + fn2)
        f1_2 = safe_div(2 * precision2 * recall2, precision2 + recall2) if precision2 and recall2 else None

        question_stats.update({
            'many_aha_accuracy': safe_div(tp, total_many_aha_trials),
            'many_aha_precision': precision,
            'many_aha_recall': recall,
            'many_aha_f1': f1,
            'little_aha_accuracy': safe_div(tn, total_little_aha_trials),
            'little_aha_precision': precision2,
            'little_aha_recall': recall2,
            'no_aha_f1': f1_2
        })
    elif all_many_aha:
        question_stats.update({
            'many_aha_only_accuracy': safe_div(sum(many_aha_correct), total_many_aha_trials)
        })
    elif all_little_aha:
        question_stats.update({
            'little_aha_only_accuracy': safe_div(sum(little_aha_correct), total_little_aha_trials)
        })
    return question_stats

all_trial_data = []
level_dfs = {}

for level in range(1, 6):
    with open(f'/home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct-thought/levels/level{level}.json', 'r') as f:
        data = json.load(f)

    level_data = []
    for row in data:
        level_data.append(analyze_row(row, level))
    
    level_df = pd.DataFrame(level_data)
    level_dfs[level] = level_df

    level_mixed = level_df[level_df['mixed_aha']]
    level_all_many_aha = level_df[level_df['all_many_aha']]
    level_all_little_aha = level_df[level_df['all_little_aha']]

    print(f"Level {level}:")
    print("Mixed:")
    print("  Avg many aha accuracy:", level_mixed['many_aha_accuracy'].mean())
    print("  Avg many aha precision:", level_mixed['many_aha_precision'].mean())
    print("  Avg many aha recall:", level_mixed['many_aha_recall'].mean())
    print("  Avg many aha f1:", level_mixed['many_aha_f1'].mean())
    print("  Avg little aha accuracy:", level_mixed['little_aha_accuracy'].mean())
    print("  Avg little aha precision:", level_mixed['little_aha_precision'].mean())
    print("  Avg little aha recall:", level_mixed['little_aha_recall'].mean())
    print("  Avg little aha f1:", level_mixed['little_aha_f1'].mean())

    print("All 10 many aha:")
    print("  Avg accuracy:", level_all_many_aha['many_aha_only_accuracy'].mean())

    print("All 10 little aha:")
    print("  Avg accuracy:", level_all_little_aha['little_aha_only_accuracy'].mean())

    # Save level-specific CSV
    level_csv_path = f'/home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct-thought/level{level}_analysis.csv'
    level_df.to_csv(level_csv_path, index=False)
    print(f"Level {level} analysis saved to {level_csv_path}")
    
    # Add to all trial data
    all_trial_data.extend(level_data)

# Calculate combined metrics
all_df = pd.DataFrame(all_trial_data)
all_mixed = all_df[all_df['mixed_aha']]
all_many_aha = all_df[all_df['all_many_aha']]
all_little_aha = all_df[all_df['all_little_aha']]

print("\nCombined analysis saved to /home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct-thought/all_levels_analysis.csv")
print("Mixed:")
print("  Avg many aha accuracy:", all_mixed['many_aha_accuracy'].mean())
print("  Avg many aha precision:", all_mixed['many_aha_precision'].mean())
print("  Avg many aha recall:", all_mixed['many_aha_recall'].mean())
print("  Avg many aha f1:", all_mixed['many_aha_f1'].mean())
print("  Avg little aha accuracy:", all_mixed['little_aha_accuracy'].mean())
print("  Avg little aha precision:", all_mixed['little_aha_precision'].mean())
print("  Avg little aha recall:", all_mixed['little_aha_recall'].mean())
print("  Avg little aha f1:", all_mixed['little_aha_f1'].mean())
print("All 10 many aha:")
print("  Avg accuracy:", all_many_aha['many_aha_only_accuracy'].mean())
print("All 10 little aha:")
print("  Avg accuracy:", all_little_aha['little_aha_only_accuracy'].mean())

# Save combined analysis
all_df.to_csv('/home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct-thought/all_levels_analysis.csv', index=False)

Level 1:
Mixed:
  Avg many aha accuracy: 1.0
  Avg many aha precision: 1.0
  Avg many aha recall: 1.0
  Avg many aha f1: 1.0
  Avg little aha accuracy: 1.0
  Avg little aha precision: 1.0
  Avg little aha recall: 1.0
  Avg little aha f1: nan
All 10 many aha:
  Avg accuracy: 0.0
All 10 little aha:
  Avg accuracy: 0.8333333333333334
Level 1 analysis saved to /home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct-thought/level1_analysis.csv
Level 2:
Mixed:
  Avg many aha accuracy: 0.9090909090909091
  Avg many aha precision: 0.9090909090909091
  Avg many aha recall: 0.9090909090909091
  Avg many aha f1: 1.0
  Avg little aha accuracy: 0.9090909090909091
  Avg little aha precision: 0.9090909090909091
  Avg little aha recall: 0.9090909090909091
  Avg little aha f1: nan
All 10 many aha:
  Avg accuracy: 0.45
All 10 little aha:
  Avg accuracy: 1.0
Level 2 analysis saved to /home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct-thought/level2_analysis.csv
Level 3:
M

#fall and spike

In [26]:
import json

#
# target_tokens = ['and', 'me', 'so', 'let', 'that', 'by', 'wait', 'but', 'think']
target_tokens = ['that', 'so', 'let', 'me', 'by']

# Load the data
with open('/home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct/top_fall_and_spike_tokens.json', 'r') as f:
    data = json.load(f)

# Print header
print(f"{'Token':<10} {'Rank':<6} {'Middle Count':<12}")
print("-" * 40)

# Extract and print info for each target token
for token in target_tokens:
    if token in data:
        info = data[token]
        print(f"{token:<10} {info['rank']:<6} {info['count']:<12}")
    else:
        print(f"{token:<10} {'N/A':<6} {'N/A':<12}")

Token      Rank   Middle Count
----------------------------------------
that       11     48          
so         9      50          
let        10     49          
me         8      52          
by         13     30          


In [27]:
import pandas as pd
import json
import matplotlib.pyplot as plt

def analyze_row(row, level):
    trials = sum(1 for key in row.keys() if key.startswith('Output_'))
    question_id = row.get('id', 'unknown')
    
    token_counts = {
        'and': 0,
        'me': 0,
        'so': 0,
        'let': 0,
        'that': 0,
        'by': 0,
        'wait': 0,
        'but': 0,
        'think': 0,
    }
    
    # Lists to store correctness for aha and no-aha cases
    aha_correct = []
    no_aha_correct = []
    total_aha_trials = 0
    total_no_aha_trials = 0
    
    # Analyze each trial
    for i in range(trials):
        output = row.get(f'Output_{i}', '').lower()
        # Count individual tokens
        for token in token_counts.keys():
            token_counts[token] += output.count(token)
            
        aha_count = sum(output.count(word) for word in token_counts.keys())
        metrics = row.get(f'Metrics_{i}', {})
        is_correct = metrics.get('math_equal', False)
        
        if aha_count > 0:
            total_aha_trials += 1
            aha_correct.append(is_correct)
        else:
            total_no_aha_trials += 1
            no_aha_correct.append(is_correct)
    
    # Add token frequencies to question_stats
    question_stats = {
        'question_id': question_id,
        'level': level,
        'total_aha_trials': total_aha_trials,
        'total_no_aha_trials': total_no_aha_trials,
        'aha_correct_count': sum(aha_correct),
        'no_aha_correct_count': sum(no_aha_correct),
        'all_aha': (total_aha_trials == trials),
        'no_aha': (total_no_aha_trials == trials),
        'mixed_aha': (not (total_aha_trials == trials)) and (not (total_no_aha_trials == trials))
    }
    # Add token counts to stats
    question_stats.update(token_counts)
    
    return question_stats, token_counts

# Collect token frequencies across all levels
total_token_counts = {
        'and': 0,
        'me': 0,
        'so': 0,
        'let': 0,
        'that': 0,
        'by': 0,
        'wait': 0,
        'but': 0,
        'think': 0,
    }

# Process each level
for level in range(1, 6):
    level_token_counts = {
        'and': 0,
        'me': 0,
        'so': 0,
        'let': 0,
        'that': 0,
        'by': 0,
        'wait': 0,
        'but': 0,
        'think': 0,
    }
    # Load and process data
    with open(f'/home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct/levels/level{level}.json', 'r') as f:
        data = json.load(f)
    
    level_data = []
    for row in data:
        stats, tokens = analyze_row(row, level)
        level_data.append(stats)
        # Update token counts
        for token, count in tokens.items():
            level_token_counts[token] += count
            total_token_counts[token] += count
    
    # Create bar plot for this level
    plt.figure(figsize=(10, 6))
    plt.bar(level_token_counts.keys(), level_token_counts.values())
    plt.title(f'Token Frequencies - Level {level}')
    plt.xlabel('Tokens')
    plt.ylabel('Frequency')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(f'/home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct/level{level}_token_frequencies.png')
    plt.close()

# Create overall bar plot
plt.figure(figsize=(10, 6))
plt.bar(total_token_counts.keys(), total_token_counts.values())
plt.title('Overall Token Frequencies')
plt.xlabel('Tokens')
plt.ylabel('Frequency')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('/home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct/total_token_frequencies.png')
plt.close()

# Print token frequencies
print("\nToken Frequencies by Level:")
for level in range(1, 6):
    print(f"\nLevel {level}:")
    for token, count in level_token_counts.items():
        print(f"{token}: {count}")

print("\nTotal Token Frequencies:")
for token, count in total_token_counts.items():
    print(f"{token}: {count}")


Token Frequencies by Level:

Level 1:
and: 7460
me: 7338
so: 11930
let: 3616
that: 5068
by: 1799
wait: 4166
but: 4014
think: 856

Level 2:
and: 7460
me: 7338
so: 11930
let: 3616
that: 5068
by: 1799
wait: 4166
but: 4014
think: 856

Level 3:
and: 7460
me: 7338
so: 11930
let: 3616
that: 5068
by: 1799
wait: 4166
but: 4014
think: 856

Level 4:
and: 7460
me: 7338
so: 11930
let: 3616
that: 5068
by: 1799
wait: 4166
but: 4014
think: 856

Level 5:
and: 7460
me: 7338
so: 11930
let: 3616
that: 5068
by: 1799
wait: 4166
but: 4014
think: 856

Total Token Frequencies:
and: 16801
me: 21985
so: 29677
let: 10086
that: 14876
by: 7127
wait: 8777
but: 8543
think: 3234


In [28]:
import pandas as pd
import json

def analyze_row(row, level):
    trials = sum(1 for key in row.keys() if key.startswith('Output_'))
    question_id = row.get('id', 'unknown')
    
    aha_correct = []
    no_aha_correct = []
    total_aha_trials = 0
    total_no_aha_trials = 0
    
    for i in range(trials): 
        output = row.get(f'Output_{i}', '').lower()
        aha_count = sum(output.count(word) for word in target_tokens)
        metrics = row.get(f'Metrics_{i}', {})
        is_correct = metrics.get('math_equal', False)
        
        if aha_count > 0:
            total_aha_trials += 1
            aha_correct.append(is_correct)
        else:
            total_no_aha_trials += 1
            no_aha_correct.append(is_correct)
    
    all_aha = (total_aha_trials == trials)
    no_aha = (total_no_aha_trials == trials)
    mixed_aha = (not all_aha) and (not no_aha)

    question_stats = {
        'question_id': question_id,
        'level': level,
        'total_aha_trials': total_aha_trials,
        'total_no_aha_trials': total_no_aha_trials,
        'aha_correct_count': sum(aha_correct),
        'no_aha_correct_count': sum(no_aha_correct),
        'all_aha': all_aha,
        'no_aha': no_aha,
        'mixed_aha': mixed_aha,
        # Initialize metrics columns with None
        'aha_accuracy': None,
        'aha_precision': None,
        'aha_recall': None,
        'aha_f1': None,
        'no_aha_accuracy': None,
        'no_aha_precision': None,
        'no_aha_recall': None,
        'no_aha_f1': None,
        'aha_only_accuracy': None,
        'no_aha_only_accuracy': None
    }

    def safe_div(n, d): return n / d if d > 0 else None

    if mixed_aha:
        tp = sum(aha_correct)
        tn = sum(no_aha_correct)
        fp = total_aha_trials - tp
        fn = total_no_aha_trials - tn

        precision = safe_div(tp, tp + fp)
        recall = safe_div(tp, tp + fn)
        f1 = safe_div(2 * precision * recall, precision + recall) if precision and recall else None
        
        tp2 = sum(aha_correct)
        tn2 = sum(no_aha_correct)
        fp2 = total_aha_trials - tp2
        fn2 = total_no_aha_trials - tn2

        precision2 = safe_div(tp2, tp2 + fp2)
        recall2 = safe_div(tp2, tp2 + fn2)
        f1_2 = safe_div(2 * precision2 * recall2, precision2 + recall2) if precision2 and recall2 else None

        question_stats.update({
            'aha_accuracy': safe_div(tp, total_aha_trials),
            'aha_precision': precision,
            'aha_recall': recall,
            'aha_f1': f1,
            'no_aha_accuracy': safe_div(tn, total_no_aha_trials),
            'no_aha_precision': precision2,
            'no_aha_recall': recall2,
            'no_aha_f1': f1_2
        })
    elif all_aha:
        question_stats.update({
            'aha_only_accuracy': safe_div(sum(aha_correct), total_aha_trials)
        })
    elif no_aha:
        question_stats.update({
            'no_aha_only_accuracy': safe_div(sum(no_aha_correct), total_no_aha_trials)
        })
    return question_stats

all_trial_data = []
level_dfs = {}

for level in range(1, 6):
    with open(f'/home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct/levels/level{level}.json', 'r') as f:
        data = json.load(f)

    level_data = []
    for row in data:
        level_data.append(analyze_row(row, level))
    
    level_df = pd.DataFrame(level_data)
    level_dfs[level] = level_df

    level_mixed = level_df[level_df['mixed_aha']]
    level_all_aha = level_df[level_df['all_aha']]
    level_no_aha = level_df[level_df['no_aha']]

    print(f"Level {level}:")
    print("Mixed:")
    print("  Avg aha accuracy:", level_mixed['aha_accuracy'].mean())
    print("  Avg aha precision:", level_mixed['aha_precision'].mean())
    print("  Avg aha recall:", level_mixed['aha_recall'].mean())
    print("  Avg aha f1:", level_mixed['aha_f1'].mean())
    print("  Avg no-aha accuracy:", level_mixed['no_aha_accuracy'].mean())
    print("  Avg no-aha precision:", level_mixed['no_aha_precision'].mean())
    print("  Avg no-aha recall:", level_mixed['no_aha_recall'].mean())
    print("  Avg no-aha f1:", level_mixed['no_aha_f1'].mean())

    print("All 10 aha:")
    print("  Avg accuracy:", level_all_aha['aha_only_accuracy'].mean())

    print("All 10 no-aha:")
    print("  Avg accuracy:", level_no_aha['no_aha_only_accuracy'].mean())

    # Save level-specific CSV
    level_csv_path = f'/home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct/level{level}_analysis.csv'
    level_df.to_csv(level_csv_path, index=False)
    print(f"Level {level} analysis saved to {level_csv_path}")
    
    # Add to all trial data
    all_trial_data.extend(level_data)

# Calculate combined metrics
all_df = pd.DataFrame(all_trial_data)
all_mixed = all_df[all_df['mixed_aha']]
all_aha = all_df[all_df['all_aha']]
all_no_aha = all_df[all_df['no_aha']]

print("\nCombined analysis saved to /home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct/all_levels_analysis.csv")
print("Mixed:")
print("  Avg aha accuracy:", all_mixed['aha_accuracy'].mean())
print("  Avg aha precision:", all_mixed['aha_precision'].mean())
print("  Avg aha recall:", all_mixed['aha_recall'].mean())
print("  Avg aha f1:", all_mixed['aha_f1'].mean())
print("  Avg no-aha accuracy:", all_mixed['no_aha_accuracy'].mean())
print("  Avg no-aha precision:", all_mixed['no_aha_precision'].mean())
print("  Avg no-aha recall:", all_mixed['no_aha_recall'].mean())
print("  Avg no-aha f1:", all_mixed['no_aha_f1'].mean())
print("All 10 aha:")
print("  Avg accuracy:", all_aha['aha_only_accuracy'].mean())
print("All 10 no-aha:")
print("  Avg accuracy:", all_no_aha['no_aha_only_accuracy'].mean())

# Save combined analysis
all_df.to_csv('/home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct/all_levels_analysis.csv', index=False)

Level 1:
Mixed:
  Avg aha accuracy: nan
  Avg aha precision: nan
  Avg aha recall: nan
  Avg aha f1: nan
  Avg no-aha accuracy: nan
  Avg no-aha precision: nan
  Avg no-aha recall: nan
  Avg no-aha f1: nan
All 10 aha:
  Avg accuracy: 0.8181818181818182
All 10 no-aha:
  Avg accuracy: nan
Level 1 analysis saved to /home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct/level1_analysis.csv
Level 2:
Mixed:
  Avg aha accuracy: nan
  Avg aha precision: nan
  Avg aha recall: nan
  Avg aha f1: nan
  Avg no-aha accuracy: nan
  Avg no-aha precision: nan
  Avg no-aha recall: nan
  Avg no-aha f1: nan
All 10 aha:
  Avg accuracy: 0.9159999999999999
All 10 no-aha:
  Avg accuracy: nan
Level 2 analysis saved to /home/mjgwak/search-o1-dev/scripts/outputs/math500.ds-qwen-14b.direct/level2_analysis.csv
Level 3:
Mixed:
  Avg aha accuracy: nan
  Avg aha precision: nan
  Avg aha recall: nan
  Avg aha f1: nan
  Avg no-aha accuracy: nan
  Avg no-aha precision: nan
  Avg no-aha recall: nan
  Avg n